In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OrdinalEncoder

# --- Configuración (un solo lugar para ajustar parámetros) ---
RUTA_CSV = "nexora_transacciones_sinteticas.csv"
CONTAMINATION = 0.03   # ATENCIÓN: 0.03 coincide con la tasa real de anomalías del dataset sintético.
                       # En producción esa tasa se desconoce: estímese con negocio/auditoría.
RANDOM_STATE = 42
COL_CATEGORICAS = ["tipo_transaccion", "dia_semana"]
COL_NUMERICAS = ["monto", "hora_dia", "destino_nuevo",
                 "antiguedad_cuenta_dias", "promedio_monto_ultimos_30_dias"]

df = pd.read_csv(RUTA_CSV)

# Validación de esquema y calidad de datos (falla temprano y con mensaje claro)
requeridas = set(COL_CATEGORICAS + COL_NUMERICAS + ["transaccion_id", "es_anomalia_real"])
faltantes = requeridas - set(df.columns)
if faltantes:
    raise ValueError(f"Faltan columnas en el CSV: {sorted(faltantes)}")
if df[COL_CATEGORICAS + COL_NUMERICAS].isna().any().any():
    raise ValueError("El CSV contiene valores nulos en columnas del modelo; limpiar antes de continuar.")

# Convertir texto a números (Isolation Forest solo entiende números).
# OrdinalEncoder reemplaza a LabelEncoder (que está pensado para etiquetas, no para features)
# y da los mismos códigos; además tolera categorías nuevas en producción (-1).
codificador = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[["tipo_transaccion_num", "dia_semana_num"]] = codificador.fit_transform(df[COL_CATEGORICAS])

# Columnas que el modelo SÍ puede ver (nunca 'es_anomalia_real')
columnas_modelo = [
    "monto", "hora_dia", "destino_nuevo",
    "antiguedad_cuenta_dias", "promedio_monto_ultimos_30_dias",
    "tipo_transaccion_num", "dia_semana_num"
]
assert "es_anomalia_real" not in columnas_modelo  # guardia contra fuga de etiquetas

X = df[columnas_modelo]

modelo = IsolationForest(contamination=CONTAMINATION, random_state=RANDOM_STATE)
df["prediccion"] = modelo.fit_predict(X)
# El modelo devuelve -1 para anomalía, 1 para normal. Lo convertimos a 0/1 para comparar fácil:
df["prediccion"] = df["prediccion"].map({1: 0, -1: 1})
# Puntaje continuo (mayor = más anómalo), útil para priorizar alertas
df["score_anomalia"] = -modelo.score_samples(X)

print(df[["transaccion_id", "monto", "es_anomalia_real", "prediccion"]].head(10))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

print(classification_report(df["es_anomalia_real"], df["prediccion"], zero_division=0))
print("Matriz de confusión [[TN, FP], [FN, TP]]:")
print(confusion_matrix(df["es_anomalia_real"], df["prediccion"]))
# Con clases desbalanceadas (3 %), la exactitud engaña: PR-AUC es más informativa.
print("PR-AUC (average precision):", round(average_precision_score(df["es_anomalia_real"], df["score_anomalia"]), 4))